# SPLADE++ (2021)
---
[[paper]](https://arxiv.org/abs/2109.10086)<br>
SPLADE = Sparse Lexical and Expansion Model

__SPLADE++__ — это метод Learned Sparse Retrieval, который трансформирует текст в высокоразмерные разреженные векторы (размерности словаря). Он совмещает преимущества традиционного поиска по ключевым словам (эффективность Inverted Index) и нейросетевых моделей (учет контекста и семантическое расширение запроса).

__Постановка задачи__<br>
Построение эффективного First Stage Retrieval. Необходимо отобразить документ $D$ и запрос $Q$ в разреженные векторы весов токенов из словаря $V$, чтобы релевантность вычислялась как скалярное произведение $S(Q, D) = \sum_{t \in Q \cap D} w_t^{(Q)} \cdot w_t^{(D)}$.

__Мотивация__<br>
Dense Retrieval (например, DPR, 2020) требует больших вычислительных ресурсов для хранения и поиска по векторным индексам (FAISS) и страдает от проблемы "максимального внутреннего произведения" (MIPS). Классический BM25 не учитывает семантику (проблема Vocabulary Mismatch). Идеальное решение должно использовать структуру Inverted Index, но уметь "догадываться", какие слова важны в контексте и какие синонимы (Expansion) стоит добавить в документ, которых там изначально не было.

__Существующие подходы__<br>
- BM25 (1990s): использует точные совпадения и статистики частот, не понимает синонимы.
- Doc2query (2019): расширяет документ, генерируя возможные вопросы с помощью T5. Это дорого вычислительно и может вносить шум.
- DeepCT (2019): перевзвешивает существующие термы через BERT, но не умеет добавлять новые слова (нет Expansion).
- SPLADE v1 (2021): первая итерация метода, использующая логарифмическую активацию, но имевшая проблемы со стабильностью обучения и качеством ранжирования без дистилляции.

__Идея__<br>
Использовать Masked Language Modeling (MLM) голову предобученного Трансформера для предсказания важности каждого слова из всего словаря для данного текста. Если модель видит текст про "смартфон", она должна выставить высокие веса не только слову "смартфон", но и словам "телефон", "гаджет", "iphone", даже если их нет в исходном тексте. Чтобы вектор оставался разреженным (sparse), применяется специальная регуляризация.

__Архитектура__<br>
В основе лежит BERT-like энкодер.
1. Input: Последовательность токенов.
2. Contextualized Embeddings: Получение векторов $h_i$ для каждого токена.
3. Logit Prediction: Для каждого токена $i$ на позиции в тексте вычисляется вес по всему словарю $V$ через веса MLM-головы: $w_{ij} = \text{transform}(h_i)^T \cdot e_j + b_j$, где $e_j$ — эмбеддинг токена $j$ из словаря.
4. Pooling & Saturation: Веса агрегируются по всей длине текста. Важнейший элемент — функция активации:
   $w_j = \sum_{i \in \text{sequence}} \log(1 + \text{relu}(w_{ij}))$
   Использование $\log$ подавляет доминирование слишком частых токенов и делает распределение весов более устойчивым.
5. Sparsification: Применение FLOPs Regularization во время обучения зануляет веса большинства нерелевантных токенов словаря.

__Алгоритм обучения__<br>
SPLADE++ ввел несколько критических улучшений в процесс обучения:
1. Contrastive Learning: Использование In-batch negatives и Hard Negatives (отбираемых с помощью других моделей ранжирования).
2. Knowledge Distillation: Обучение на "мягких" метках от мощного Cross-Encoder (например, на базе Electra или RoBERTa). Модель учится воспроизводить скоры релевантности, а не просто бинарную метку.
3. Loss Function: Комбинация Ranking Loss (например, MarginMSE) и Regularization Loss (FLOPs penalty).
   $\mathcal{L} = \mathcal{L}_{distill} + \lambda \cdot \mathcal{L}_{flops}$
   $\mathcal{L}_{flops}$ минимизирует ожидаемое количество операций при поиске, заставляя модель выбирать только самые информативные токены.

__Алгоритм инференса__<br>
1. Indexing (Offline): Документы прогоняются через SPLADE++, на выходе — разреженные векторы (обычно < 100-200 ненулевых компонент из 30 000). Эти веса сохраняются в стандартный Inverted Index (например, Lucene/Elasticsearch).
2. Query Encoding (Online): Запрос преобразуется в такой же разреженный вектор.
3. Retrieval: Выполняется стандартный поиск по инвертированному индексу. Поскольку векторы разрежены, скорость сопоставима с BM25.

__Результаты__<br>
Эксперименты на датасете MS MARCO (Passage Ranking):
- SPLADE++ достиг MRR@10 = 0.380, что на тот момент превосходило лучшие Dense Retrieval модели (DPR ~0.320) и значительно обходило BM25 (0.184).
- В отличие от Dense моделей, SPLADE++ показал значительно лучшую обобщающую способность на Zero-shot задачах (бенчмарк BEIR), так как сохраняет лексическую точность.
- По сравнению с DeepCT, точность Recall@1000 выросла более чем на 10% за счет механизма Expansion.

## 📝 Критический анализ

```markdown
# SPLADE++ (2021)
---
[[paper]](https://arxiv.org/abs/2109.10086)<br>
SPLADE = Sparse Lexical and Expansion Model

**SPLADE++** — метод Learned Sparse Retrieval, преобразующий текст в разреженные векторы. Он сочетает эффективность Inverted Index и семантическое расширение запроса нейросетями.

__Постановка задачи__<br>
Создание эффективного First Stage Retrieval. Необходимо отобразить документ $D$ и запрос $Q$ в разреженные векторы, чтобы релевантность вычислялась как скалярное произведение.

__Мотивация__<br>
Dense Retrieval требует больших ресурсов и страдает от MIPS. BM25 не учитывает семантику. Идеальное решение использует Inverted Index и семантическое расширение.

__Существующие подходы__<br>
- BM25: точные совпадения, не понимает синонимы.
- Doc2query: расширяет документ, но дорого и шумно.
- DeepCT: перевзвешивает термы, но без Expansion.
- SPLADE v1: логарифмическая активация, но проблемы со стабильностью.

__Идея__<br>
Использовать Masked Language Modeling для предсказания важности слов. Регуляризация сохраняет разреженность векторов.

__Архитектура__<br>
BERT-like энкодер:
1. Input: Токены.
2. Contextualized Embeddings: Векторы $h_i$.
3. Logit Prediction: Веса через MLM-голову.
4. Pooling & Saturation: Агрегация весов.
5. Sparsification: FLOPs Regularization зануляет нерелевантные веса.

<img src="img/img.png" width=500>

__Алгоритм обучения__<br>
1. Contrastive Learning: In-batch и Hard Negatives.
2. Knowledge Distillation: Обучение на "мягких" метках.
3. Loss Function: Ranking и Regularization Loss.

__Алгоритм инференса__<br>
1. Indexing: Документы в разреженные векторы.
2. Query Encoding: Запрос в разреженный вектор.
3. Retrieval: Поиск по инвертированному индексу.

__Результаты__<br>
На MS MARCO SPLADE++ достиг MRR@10 = 0.380, превосходя Dense Retrieval модели и BM25. Обладает лучшей обобщающей способностью на Zero-shot задачах и улучшает Recall@1000 более чем на 10% по сравнению с DeepCT.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации основных концепций SPLADE++ на Python

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel

# SPLADE++ использует BERT-like энкодер для получения контекстуализированных эмбеддингов
class SPLADEPlusPlus(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased'):
        super(SPLADEPlusPlus, self).__init__()
        self.tokenizer = BertTokenizer.from_pretrained(bert_model_name)
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.vocab_size = self.bert.config.vocab_size
        self.linear = nn.Linear(self.bert.config.hidden_size, self.vocab_size)

    def forward(self, input_text):
        # Токенизация входного текста
        inputs = self.tokenizer(input_text, return_tensors='pt', padding=True, truncation=True)
        # Получение контекстуализированных эмбеддингов
        outputs = self.bert(**inputs)
        # Применение линейного слоя для предсказания весов по всему словарю
        logits = self.linear(outputs.last_hidden_state)
        # Агрегация весов по всей длине текста с использованием логарифмической активации
        weights = torch.log1p(F.relu(logits)).sum(dim=1)
        return weights

# Пример использования модели для преобразования текста в разреженный вектор
model = SPLADEPlusPlus()

# Пример документа и запроса
document = "The smartphone has a great camera and long battery life."
query = "Best phone with good camera"

# Получение разреженных векторов для документа и запроса
doc_weights = model(document)
query_weights = model(query)

# Вычисление релевантности как скалярное произведение
relevance_score = torch.dot(doc_weights.squeeze(), query_weights.squeeze())

print(f"Relevance Score: {relevance_score.item()}")

# Пример регуляризации FLOPs для разреживания векторов
def flops_regularization(weights, lambda_flops=0.1):
    # FLOPs penalty для зануления нерелевантных токенов
    return lambda_flops * torch.sum(weights > 0).float()

# Пример вычисления FLOPs регуляризации
flops_penalty = flops_regularization(doc_weights)
print(f"FLOPs Penalty: {flops_penalty.item()}")

# Пример обучения с использованием контрастивного обучения и дистилляции
def training_step(model, document, query, cross_encoder_score, lambda_flops=0.1):
    # Получение разреженных векторов
    doc_weights = model(document)
    query_weights = model(query)
    
    # Вычисление релевантности
    relevance_score = torch.dot(doc_weights.squeeze(), query_weights.squeeze())
    
    # Loss функции: дистилляция и регуляризация
    distillation_loss = F.mse_loss(relevance_score, cross_encoder_score)
    flops_penalty = flops_regularization(doc_weights, lambda_flops)
    
    # Общая функция потерь
    loss = distillation_loss + flops_penalty
    return loss

# Пример использования функции обучения
cross_encoder_score = torch.tensor(0.8)  # Пример "мягкой" метки от Cross-Encoder
loss = training_step(model, document, query, cross_encoder_score)
print(f"Training Loss: {loss.item()}")
```

Этот код иллюстрирует основные концепции SPLADE++, такие как использование BERT-like энкодера для получения контекстуализированных эмбеддингов, применение логарифмической активации для агрегации весов, и использование FLOPs регуляризации для разреживания векторов. Также показан пример обучения с использованием дистилляции и контрастивного обучения.